In [1]:
import json

def Load_josn(p):
    with open(p, 'r', encoding='utf-8') as f:
        d = json.load(f)
    return d


pdf_urls=Load_josn('data_set\\pdf_urls.json')
question_doc = Load_josn('data_set\\qrels.json')
queries=Load_josn('data_set\\queries.json')
answers=Load_josn('data_set\\answers.json')



In [2]:
import pandas as pd
df = pd.DataFrame.from_dict(question_doc, orient='index')
df.index.name = 'query_id'
df = df.reset_index()  
df.head(2)

,query_id,doc_id,section_id
0,852703f0-8373-43a2-a18a-eb5908ad0779,2410.14077v2,1
1,9199173b-3ed1-4118-88cd-1713fc5fa8a7,2404.00822v2,17


In [7]:
doc_list=df.groupby('doc_id').count().sort_values(by='query_id',ascending=False).head(10).index.tolist()
df=df[df['doc_id'].isin(doc_list)]

### To Download the PDFs use the below block

In [ ]:
# import requests, os
# from concurrent.futures import ThreadPoolExecutor, as_completed

# os.makedirs("pdfs", exist_ok=True)
# items ={}
# for i ,j in pdf_urls.items():
#     if(i in doc_list):
#         items[i]=j
    
# def download(name_url):
#     name, url = name_url
#     try:
#         fname = os.path.join("pdfs", f"{name}.pdf")
        
#         res = requests.get(url, timeout=30)
#         res.raise_for_status()
#         with open(fname, "wb") as f:
#             f.write(res.content)
#         return name, True, None
#     except Exception as e:
#         return name, False, str(e)

# pdfs_ = []
# with ThreadPoolExecutor(max_workers=8) as executor:
#     futures = [executor.submit(download, item) for item in items.items()]
#     for future in as_completed(futures):
#         name, ok, err = future.result()
#         if ok:
#             pdfs_.append(name)
#         else:
#             print(f"failed: {name} -> {err}")

# print(f"{len(pdfs_)}/{len(items)} downloaded")

In [3]:
q_df = pd.DataFrame.from_dict(queries, orient='index')  # columns: query, type, source
q_df.index.name = 'query_id'
q_df = q_df.reset_index().rename(columns={'type': 'query_type', 'source': 'query_source', 'query': 'query'})

df = df.merge(q_df, on='query_id', how='left')

In [4]:
answer_df=pd.DataFrame.from_dict(answers,orient="index")
answer_df.index.name='query_id'
df=df.merge(answer_df,on='query_id',how='left')

In [6]:
df=df.rename(columns={0:'answer'})
df.head(2)


,query_id,doc_id,section_id,query,query_type,query_source,answer
0,852703f0-8373-43a2-a18a-eb5908ad0779,2410.14077v2,1,What are the challenges in estimating output i...,abstractive,text-image,Estimating output impedance in inverter-based ...
1,9199173b-3ed1-4118-88cd-1713fc5fa8a7,2404.00822v2,17,How do changes in effective microbial death ra...,abstractive,text,Increases in heterogeneity related to effectiv...


In [7]:
from rag_eval.parsers import PDFPlumberParser 
p=PDFPlumberParser()
d=p.parse('D:\\projects\\RAG-Chunking-Eval\\pdfs\\2401.03305v2.pdf')
d.pages[0].content[:200]

'Leveraging IS and TC: Optimal order execution subject to\nreference strategies\nXue Cheng1, Peng Guo1, and Tai-Ho Wang2\n1\nLMEQF,DepartmentofFinancialMathematics,SchoolofMathematicalSciences,PekingUniver'

In [ ]:
from rag_eval.chunkers import RecursiveChunker
r=RecursiveChunker()
chunks_=r.chunk(d)
chunks_[0].text[:200]

In [ ]:
from rag_eval.embedders import HuggingFaceEmbedder
he=HuggingFaceEmbedder()
chunks_df=pd.DataFrame(columns=['doc_id','doc_path','chunk','page_no','chunk_no','embedding'])
chunk_data=[]

for i in chunks_:
    chunk_data.append({'doc_id':i.source_id,'doc_path':'','page_no':i.page_numbers,'chunk':i.text,'chunk_no':i.chunk_no,'embedding':he.embed_query(i.text)})


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5750.06it/s]


In [14]:
q1_df=df[df['doc_id']=='2401.03305v2']
q1_df["embedding"] = q1_df.apply(
    lambda x: he.embed_query(x["query"]),
    axis=1
)

C:\Users\vishw\AppData\Local\Temp\ipykernel_8976\405649787.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  q1_df["embedding"] = q1_df.apply(


In [15]:
q1_df.columns

Index(['query_id', 'doc_id', 'section_id', 'query', 'query_type',
       'query_source', 'answer', 'embedding'],
      dtype='object')

In [16]:
chunks_df.columns

Index(['doc_id', 'doc_path', 'chunk', 'page_no', 'chunk_no', 'embedding'], dtype='object')

In [17]:
q1_df.head(1)

,query_id,doc_id,section_id,query,query_type,query_source,answer,embedding
107,0920cb6c-229b-4b46-b2ab-834dffea6689,2401.03305v2,2,How do implementation shortfall (IS) and targe...,abstractive,text,Implementation shortfall (IS) orders aim to ex...,"[-0.031133107841014862, 0.07394389063119888, 0..."


In [18]:
chunks_df=pd.DataFrame(chunk_data)

In [19]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Precompute chunk embeddings ONCE outside the function
chunk_embs = np.vstack(chunks_df['embedding'].values)  # More robust than tolist()

def get_top_5_chunks(row):
    """Find top 5 most similar chunks for a query."""
    query_emb = np.array(row['embedding']).reshape(1, -1)
    
    # Cosine similarity: shape (1, num_chunks)
    similarities = cosine_similarity(query_emb, chunk_embs)[0]
    
    # Get indices of top 5
    top_indices = np.argsort(similarities)[-5:][::-1]
    
    # Return as list of dicts with chunk info + similarity score
    return [
        {
            'chunk': chunks_df.iloc[idx]['chunk'],
            'doc_id': chunks_df.iloc[idx]['doc_id'],
            'page_no': chunks_df.iloc[idx]['page_no'],
            'chunk_no': chunks_df.iloc[idx]['chunk_no'],
            'similarity': float(similarities[idx]),
        }
        for idx in top_indices
    ]

q1_df['top_5'] = q1_df.apply(get_top_5_chunks, axis=1)

C:\Users\vishw\AppData\Local\Temp\ipykernel_8976\3931682640.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  q1_df['top_5'] = q1_df.apply(get_top_5_chunks, axis=1)


In [21]:
from rag_eval.parsers import DoclingParser
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"  # must be before any torch/docling import for windows

dp=DoclingParser()
doc_=dp.parse('D:\\projects\\RAG-Chunking-Eval\\pdfs\\2401.03305v2.pdf')

[INFO] 2026-08-16 23:40:56,459 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-16 23:40:56,461 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-08-16 23:40:56,473 [RapidOCR] download_file.py:60: File exists and is valid: D:\projects\RAG-Chunking-Eval\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-16 23:40:56,473 [RapidOCR] main.py:50: Using D:\projects\RAG-Chunking-Eval\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-16 23:40:56,615 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-16 23:40:56,615 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-08-16 23:40:56,620 [RapidOCR] download_file.py:60: File exists and is valid: D:\projects\RAG-Chunking-Eval\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-16 23:40:56,620 [RapidOCR] main.py:50: Using D:\projects\RAG-Chunking-Eval\.venv\Lib\site-packages\rapidocr\models\

In [30]:
docling_chunks=r.chunk(doc_)
docling_chunks[0].text

'1\nLeveraging IS and TC: Optimal order execution subject to reference strategies\nXue Cheng 1 , Peng Guo 1 , and Tai-Ho Wang 2\nLMEQF, Department of Financial Mathematics, School of Mathematical Sciences, Peking University, Beijing 100871, China. 2\nDepartment of Mathematics, Baruch College, CUNY, 1 Bernard Baruch Way, New York, NY 10010, USA\nMarch 5, 2025\nAbstract'

In [31]:
docling_chunk_embedding=[he.embed_query(i.text) for i in docling_chunks  ]

In [37]:
import torch
import gc

# Force-unload any models sitting in GPU memory from earlier cells
gc.collect()
torch.cuda.empty_cache()

# If Docling parser was already initialized, delete it
try:
    del dp
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GiB")

Free VRAM: 4.3 GiB


In [ ]:
from rag_eval.interfaces import RetrievalJudgement
from rag_eval.llms import OllamaLLM


ollama_llm=OllamaLLM(model="llama3.2:3b")

RETRIEVAL_JUDGE_SYSTEM_PROMPT = """\
You are an evaluator for a Retrieval-Augmented Generation (RAG) system.
Your task is to determine whether the retrieved chunks contain sufficient \
and relevant information to answer the given query correctly.

You will receive:
* A QUERY: the question asked by the user.
* A GROUND TRUTH ANSWER: the expected answer.
* RETRIEVED CHUNKS: the documents/chunks retrieved by the RAG system.

Evaluate the retrieved chunks as a whole.

Return:
PASS if the retrieved chunks contain enough relevant information to \
derive the ground-truth answer.
FAIL if:
* The retrieved chunks are irrelevant to the query.
* The chunks do not contain the information required to answer the query.
* Important information required for the ground-truth answer is missing.
* The chunks contain information that contradicts the ground-truth answer \
and would prevent a correct answer.
* The information is too vague or incomplete to reasonably derive the \
expected answer.

Important rules:
* Judge semantic relevance, not just keyword overlap.
* Do not use external knowledge.
* Do not assume information that is not present in the retrieved chunks.
* The ground-truth answer is the reference for determining whether the \
retrieved information is sufficient.
* The retrieved chunks do not need to contain the exact wording of the \
ground-truth answer.
* Paraphrases and semantically equivalent information should be \
considered valid.
* Multiple chunks can collectively provide the required information.
* Do not judge the quality of the chunking method itself.
* Judge only whether the retrieved content is sufficient for answering \
the query."""


RETRIEVAL_JUDGE_USER_TEMPLATE = """\
Evaluate the following RAG retrieval result.

QUERY:
{query}

GROUND TRUTH ANSWER:
{ground_truth_answer}

RETRIEVED CHUNKS:
{retrieved_chunks}

Determine whether the retrieved chunks are sufficient to answer the \
query and derive the ground-truth answer."""


user_prompt = RETRIEVAL_JUDGE_USER_TEMPLATE.format(
        query=query,
        ground_truth_answer=ground_truth,
        retrieved_chunks=retrieved_chunks,
    )

res=ollama_llm.invoke_structured(user_prompt,RetrievalJudgement,system_prompt=RETRIEVAL_JUDGE_SYSTEM_PROMPT)